# CBS Match: A Python-Powered Weekly Matchmaking App for Columbia Business School

**Python WOW — Spring 2026**

---

## Inspiration

A few weeks ago, a friend sent a link to [DateDrop](https://stanford.trydatedrop.com/) — a matchmaking app built specifically for Stanford students. The concept was elegant: one match per week, revealed at the same time for everyone, no endless swiping. It got us thinking: *why doesn't CBS have something like this?*

We decided to build it.

**CBS Match** gives every MBA student a weekly match based on a 75-question personality survey. Monday morning, you and one other student each see each other's profile at the same time. You decide whether to connect.

The entire backend — personality scoring, the matching algorithm, the API, the database layer — is **written in Python**. That's what this notebook is about.

---

## The M7 Vision

We didn't build just a CBS app. We built a **multi-school platform**.

The codebase supports separate, isolated experiences for every M7 business school — each with its own email domain validation, its own matching pool, and its own admin dashboard:

| School | Tenant | Email Domain |
|---|---|---|
| Columbia Business School | `cbs` | `@gsb.columbia.edu` |
| Harvard Business School | `hbs` | `@hbs.edu` |
| Stanford GSB | `stanford` | `@gsb.stanford.edu` |
| Wharton | `wharton` | `@wharton.upenn.edu` |
| Booth | `booth` | `@chicagobooth.edu` |
| Kellogg | `kellogg` | `@kellogg.northwestern.edu` |
| MIT Sloan | `sloan` | `@sloan.mit.edu` |

Students from different schools never appear in each other's matches. A Harvard student sees only other Harvard students. Same for CBS.

For this demo, everything is under the **CBS umbrella** — all login credentials use `@gsb.columbia.edu` addresses. But the infrastructure to flip on any of the other six schools is already in place. We like that it makes the project feel bigger than any single class project — because we intend for it to be.

---

## How This Was Built

This took approximately **3 weeks** and well over **100 hours** of work. Over the course of building it, we used **more than one billion tokens** across AI tools — a number that would have seemed absurd a few years ago and now feels like a normal Thursday.

**The AI stack we used:**

- **ChatGPT** — Research and review partner for the psychology layer. We used it to evaluate our Big Five question selection against the academic literature, stress-test the weighting scheme for the compatibility score, and refine the survey copy to feel human rather than clinical.

- **Claude (Anthropic)** — Primary coding collaborator. Architecture decisions, the Python scoring engine, debugging the stable marriage implementation, reviewing every deployment configuration, and writing much of the prose throughout the app.

- **Claude Agents via Cline (VS Code)** — This is where most of the actual building happened. [Cline](https://github.com/cline/cline) is an AI coding agent that runs inside VS Code. It interfaces with Claude and acts autonomously: reading files, writing code, running terminal commands, navigating the codebase, and deploying to production. We'd describe a feature or a bug, and the agent would implement it end-to-end — reading the relevant files, writing the fix, running the tests, and committing the change. This is how we compressed what might have been months of solo work into three weeks.

**The human work:** The ideas, the product decisions, the personality framework, the UX flow, the writing, the judgment calls on what to build and what to skip — that was all us. AI dramatically accelerated execution. It did not replace thinking.

---

## Links

| | |
|---|---|
| **Live app** | https://cbs-match-web.vercel.app |
| **API (Render)** | https://cbs-match-api.onrender.com/docs |
| **GitHub repo** | https://github.com/T-J-C-99/cbs-match |

The GitHub repo is public. You can browse the full source code — the FastAPI backend, the scoring engine, the matching algorithm, the Next.js frontend, the database migrations, everything.

---

> **Note for David:** Jump to *Part 7* if you want to skip straight to the live demo and login credentials. If you run into any issues — app not loading, login not working, match not appearing — please reach out directly and we'll fix it. The Render backend runs on a free tier and may take ~30 seconds to wake up from a cold start on the first request.

---

## Part 1: The New User Experience

Before we get into code, here's what it actually looks like to sign up and use the app — because the UX shapes everything about why the backend works the way it does.

### Step 1 — Create an Account

Go to **https://cbs-match-web.vercel.app** and click *Sign Up*. You register with your CBS email (`@gsb.columbia.edu`). The system validates the domain — only CBS students can sign up for the CBS pool.

You set a display name, your CBS year ('26 or '27), your hometown, and your gender identity + who you're seeking. Photos come later.

### Step 2 — The Personality Survey (75 Questions)

Right after registration, you're taken into the survey. It has 8 screens and covers:

- **Big Five personality traits** — openness, conscientiousness, extraversion, agreeableness, emotional regulation (15 Likert questions)
- **Conflict & repair style** — how you handle tension in relationships (12 questions)
- **Attachment & communication** — secure vs. anxious vs. avoidant tendencies (9 questions)
- **Life architecture** — kids timeline, career ambition, marriage views, values (15 questions)
- **Lifestyle & finances** — social energy, home preferences, financial style (14 questions)
- **Forced-choice pairs** — 8 binary tradeoffs (stability vs. adventure, career vs. relationship, etc.)
- **Vibe questions** — ideal first date, how you decompress, communication style, favorite CBS spot

The survey takes about 10–12 minutes. Answers are saved as you go — you can leave and come back.

### Step 3 — Your Vibe Card

When you complete the survey, the backend computes your personality profile and generates a **Vibe Card** — a short, human-readable summary of who you are in a relationship context. It pulls from your Big Five scores, your conflict style, your attachment pattern, and your vibe answers to produce something that feels personal rather than algorithmic.

You see it on the completion screen. It's also what your match will see about you.

### Step 4 — Complete Your Profile

After the survey, you're prompted to finish your profile: add photos, fill in your phone number and Instagram handle if you want them visible to a match, and review your display details.

### Step 5 — Wait for Monday

Matching runs **once a week**. If you sign up on a Wednesday, you won't have a match until the following Monday. The app shows a clean "check back Monday" state on the match page.

**This is why we provide demo accounts below.** We pre-ran the matching algorithm with six personality profiles and have two live matched pairs ready to explore. You don't have to wait — you can jump straight to the match reveal experience using the credentials in Part 7.

In [ ]:
# Install dependencies if needed (standard in most Python environments)
# pip install requests matplotlib numpy

import json
import math
import requests
import matplotlib.pyplot as plt
import numpy as np

# Production endpoints
BASE = "https://cbs-match-api.onrender.com"   # Python/FastAPI backend on Render
SITE = "https://cbs-match-web.vercel.app"     # Next.js frontend on Vercel
REPO = "https://github.com/T-J-C-99/cbs-match"

# Confirm the API is up
# Note: Render's free tier spins down after inactivity.
# The first request may take ~30 seconds. Subsequent requests are fast.
print("Checking API health...")
r = requests.get(f"{BASE}/health", timeout=60)
print(f"Status: {r.status_code}")
print(f"Response: {r.json()}")
print(f"\nAPI docs available at: {BASE}/docs")

---

## Part 2: The Architecture

```
┌────────────────────────────┐        ┌──────────────────────────────────────┐
│   Next.js Frontend         │        │   FastAPI Backend (Python)           │
│   Vercel                   │ ─────▶ │   Render                             │
│                            │ ◀───── │                                      │
│  • Registration / login    │        │  • Auth, JWT tokens, refresh         │
│  • 75-question survey UI   │        │  • Personality scoring engine        │
│  • Match reveal page       │        │  • Gale-Shapley matching algorithm   │
│  • Profile + settings      │        │  • PostgreSQL via SQLAlchemy         │
│  • Safety (block, report)  │        │  • Multi-tenant isolation            │
└────────────────────────────┘        └──────────────────────────────────────┘
          │                                          │
     Vercel CDN                              PostgreSQL DB
  (auto-deploys from                     (Render managed)
    GitHub main)
```

**Key design decision:** the Next.js frontend never calls the Python backend directly from the browser. All API calls from client code go through **Next.js server-side route handlers**, which read auth from httpOnly cookies and forward requests to Render with a Bearer token. This keeps credentials off the client and avoids cross-origin cookie issues.

**Python packages doing the heavy lifting:**

| Package | Role |
|---|---|
| `fastapi` | Web framework — async, typed, auto-generates OpenAPI docs |
| `sqlalchemy` | ORM — all DB queries are type-safe Python, no raw SQL |
| `pyjwt` | JSON Web Token auth — stateless, no session table needed |
| `bcrypt` | Password hashing — salted, one-way |
| `psycopg2` | PostgreSQL driver |
| `pydantic` | Request/response validation and serialization |

---

## Part 3: Authenticating with the API

CBS Match uses **JWT (JSON Web Token)** authentication. When a user logs in, the server signs a short-lived token (15 minutes) with a secret key. A longer-lived refresh token (30 days) allows silent renewal. Neither token touches the database after creation — verification is pure cryptography.

Here's how it works in Python:

In [ ]:
def login(email: str, password: str = "community123") -> str:
    """Log in to the CBS Match API and return a bearer token."""
    response = requests.post(
        f"{BASE}/auth/login",
        json={"email": email, "password": password},
        headers={
            "X-Tenant-Slug": "cbs",     # identifies the CBS school context
            "X-Auth-Mode":  "bearer",   # return token in body (vs. set cookie)
        },
        timeout=30,
    )
    if response.status_code != 200:
        raise ValueError(f"Login failed ({response.status_code}): {response.text}")
    token = response.json()["access_token"]
    print(f"  ✓  Logged in as {email}")
    return token


def api_get(path: str, token: str) -> dict:
    """Authenticated GET against the CBS Match API."""
    response = requests.get(
        f"{BASE}{path}",
        headers={"Authorization": f"Bearer {token}", "X-Tenant-Slug": "cbs"},
        timeout=30,
    )
    return response.json()


# Try it — log in as one of our demo users
brad_token   = login("brad@gsb.columbia.edu")
brad_profile = api_get("/users/me/profile", brad_token)

print(f"\n  Name:      {brad_profile.get('display_name')}")
print(f"  Class:     CBS '{brad_profile.get('cbs_year')}")
print(f"  Hometown:  {brad_profile.get('hometown')}")
print(f"  Gender:    {brad_profile.get('gender_identity')}, seeking {brad_profile.get('seeking_genders')}")

---

## Part 4: The 75-Question Personality Survey

The survey is stored as a structured JSON definition (`questions.json`) that the frontend reads to render each screen. The backend validates answers against this same definition before saving them.

This separation — survey definition in JSON, rendering in the frontend, validation and scoring in Python — means we can iterate on survey questions without deploying backend code changes. We reviewed the question set against psychology literature with ChatGPT to make sure each dimension was covered by enough items to be reliable.

In [ ]:
# Load the survey definition (the same file served to the frontend)
with open("questions.json") as f:
    survey_def = json.load(f)

# Inventory the survey
question_types = {}
all_questions  = []

for screen in survey_def["screens"]:
    for item in screen["items"]:
        q  = item["question"]
        rt = q["response_type"]
        question_types[rt] = question_types.get(rt, 0) + 1
        all_questions.append(q)

print(f"Total questions: {len(all_questions)}")
print(f"\nBy response type:")
for qtype, count in sorted(question_types.items()):
    print(f"  {qtype:<22}  {count}")

print(f"\nScreens:")
for screen in survey_def["screens"]:
    title = screen.get("title", screen.get("key", "?"))
    print(f"  {title:<35}  {len(screen['items'])} questions")

In [ ]:
# Show one example of each question type
seen = set()
print("Sample questions by type:\n")
for q in all_questions:
    rt = q["response_type"]
    if rt not in seen:
        seen.add(rt)
        print(f"  [{rt}]")
        print(f"    {q['code']}: {q['text']}")
        print()

### The Vibe Card

When the survey is completed, the backend runs a scoring pass over the answers and saves a `user_traits` record. It also generates a **Vibe Card** — a short, personality-driven summary. Here's what one looks like for a real user:

In [ ]:
# Fetch Brad's vibe card from the API
brad_token = login("brad@gsb.columbia.edu")
vibe = api_get("/users/me/vibe-card", brad_token)

print("Brad's Vibe Card\n" + "─" * 40)
if isinstance(vibe, dict):
    # Print whatever fields the vibe card has
    for key, val in vibe.items():
        if key not in ("id", "user_id", "survey_slug", "survey_version", "computed_at"):
            if isinstance(val, list):
                print(f"\n{key}:")
                for item in val:
                    print(f"  • {item}")
            elif isinstance(val, str) and len(val) > 0:
                print(f"\n{key}: {val}")
else:
    print(vibe)

---

## Part 5: The Compatibility Scoring Engine

Once two users have completed the survey, the backend scores their compatibility across six dimensions. Each dimension is a separate Python function that compares the two users' answer vectors, produces a score between 0 and 1, and then the final score is a weighted sum.

We spent several sessions with ChatGPT reviewing how each weight should be calibrated against the relationship psychology literature. The most important finding: **conflict style compatibility and attachment security** are stronger predictors of relationship success than raw personality similarity. That's why they carry the most weight.

```
final_score = 0.22 × values_alignment
            + 0.22 × attachment_fit
            + 0.24 × conflict_style_fit
            + 0.12 × emotional_similarity
            + 0.12 × personality_fit
            + 0.08 × life_goals_fit
```

If the final score is below **0.60**, the user receives a "no match this week" result rather than a poor match. We'd rather send no match than a bad one.

In [ ]:
# The scoring formula — a simplified mirror of the Python backend

WEIGHTS = {
    "values_similarity":    0.22,
    "attachment_fit":       0.22,
    "conflict_fit":         0.24,
    "emotional_similarity": 0.12,
    "personality_fit":      0.12,
    "life_fit":             0.08,
}

def compute_score(category_scores: dict) -> float:
    """Weighted compatibility score from per-category subscores."""
    return round(sum(WEIGHTS[cat] * category_scores.get(cat, 0)
                     for cat in WEIGHTS), 4)


# Real scores from the live system — Brad & Margot, Tom & Jennifer
brad_margot_cats = {
    "values_similarity":    1.000,
    "attachment_fit":       0.875,
    "conflict_fit":         0.521,
    "emotional_similarity": 0.833,
    "personality_fit":      0.792,
    "life_fit":             0.875,
}

tom_jennifer_cats = {
    "values_similarity":    1.000,
    "attachment_fit":       0.818,
    "conflict_fit":         0.625,
    "emotional_similarity": 0.917,
    "personality_fit":      0.917,
    "life_fit":             0.781,
}

MIN_SCORE = 0.60

brad_score = compute_score(brad_margot_cats)
tom_score  = compute_score(tom_jennifer_cats)

print(f"Brad & Margot  →  {brad_score:.4f}  ({brad_score*100:.1f}%)  "
      f"{'✓ match' if brad_score >= MIN_SCORE else '✗ no match'}")
print(f"Tom & Jennifer →  {tom_score:.4f}  ({tom_score*100:.1f}%)  "
      f"{'✓ match' if tom_score >= MIN_SCORE else '✗ no match'}")

---

## Part 6: The Stable Marriage Algorithm

Scoring tells us *how compatible* two people are. But with 450 men and 450 women in the eligible pool, we can't just pair each person with their highest-scoring candidate — someone else might need that person more. We need a globally stable assignment.

This is the **Stable Marriage Problem**, solved by the **Gale-Shapley algorithm** (1962). Shapley won the Nobel Prize in Economics in 2012 for this work.

**What "stable" means:** no two unmatched people both prefer each other to their assigned partners. If such a pair existed, they'd defect — making the matching unstable.

The algorithm runs in O(n²) time — fast enough to run the entire CBS cohort in well under a second.

In [ ]:
def gale_shapley(group_a_prefs: dict, group_b_prefs: dict) -> dict:
    """
    Gale-Shapley stable matching.

    group_a_prefs: { person: [ranked_list_of_group_b_people] }
    group_b_prefs: { person: [ranked_list_of_group_a_people] }

    Returns: { group_a_person: group_b_person } for all matched pairs.
    """
    free_a       = list(group_a_prefs.keys())
    next_propose = {p: 0 for p in free_a}   # next index to propose to
    b_partner    = {}                        # current partner for each group-B person

    while free_a:
        a = free_a[0]
        b = group_a_prefs[a][next_propose[a]]
        next_propose[a] += 1

        if b not in b_partner:
            b_partner[b] = a          # B is free — accept
            free_a.pop(0)
        else:
            current = b_partner[b]
            if group_b_prefs[b].index(a) < group_b_prefs[b].index(current):
                b_partner[b] = a      # B prefers new proposer — switch
                free_a.pop(0)
                free_a.append(current)
            # else B keeps current partner; A stays free and tries next

    return {a: b for b, a in b_partner.items()}


# ── Run it with our 6 CBS students ────────────────────────────────────────────
# Preference lists ordered by compatibility score (highest = first choice)

men_prefs = {
    "Brad": ["Margot",   "Jennifer", "Oprah"],
    "Tom":  ["Jennifer", "Margot",   "Oprah"],
    "Matt": ["Oprah",    "Jennifer", "Margot"],
}
women_prefs = {
    "Jennifer": ["Tom",  "Brad", "Matt"],
    "Margot":   ["Brad", "Tom",  "Matt"],
    "Oprah":    ["Matt", "Tom",  "Brad"],
}

matches = gale_shapley(men_prefs, women_prefs)

print("Stable matching result:")
for man, woman in matches.items():
    print(f"  {man:8} ↔ {woman}")

# Verify stability: check for blocking pairs
blocking = [
    (man, other_woman)
    for man, woman in matches.items()
    for other_woman, other_man in matches.items()
    if other_woman != woman
    and men_prefs[man].index(other_woman) < men_prefs[man].index(woman)
    and women_prefs[other_woman].index(man) < women_prefs[other_woman].index(other_man)
]

print(f"\nBlocking pairs: {blocking if blocking else 'none — matching is stable ✓'}")

---

## Part 7: Live Results from the API

In [ ]:
# Fetch the current week's match for each demo user from the live production API

demo_users = [
    {"name": "Brad",     "email": "brad@gsb.columbia.edu"},
    {"name": "Margot",   "email": "margot@gsb.columbia.edu"},
    {"name": "Tom",      "email": "tom@gsb.columbia.edu"},
    {"name": "Jennifer", "email": "jennifer@gsb.columbia.edu"},
]

live_results = []
for user in demo_users:
    token   = login(user["email"])
    data    = api_get("/matches/current", token)
    match   = data.get("match", {})
    profile = match.get("matched_profile") or {}
    score   = float(match.get("score_total") or 0)
    cats    = match.get("score_breakdown", {}).get("categories", {})
    live_results.append({
        "name":        user["name"],
        "matched":     profile.get("display_name", "—"),
        "score":       score,
        "categories":  cats,
        "week":        match.get("week_start_date", "?"),
    })

print(f"\nWeek of: {live_results[0]['week']}\n")
print(f"  {'User':<12}  {'Matched With':<12}  {'Score':>6}")
print("  " + "─" * 38)
for r in live_results:
    print(f"  {r['name']:<12}  {r['matched']:<12}  {r['score']*100:>5.1f}%")

---

## Part 8: Visualizing Compatibility

The score breakdown lets us see not just *how compatible* two people are, but *where* the compatibility comes from.

In [ ]:
# ── Radar charts — one per matched pair ───────────────────────────────────────

LABELS = ["Values", "Attachment", "Conflict\nStyle", "Emotional\nFit", "Personality", "Life Goals"]
N      = len(LABELS)
angles = [n / N * 2 * math.pi for n in range(N)] + [0]   # close polygon

brad_vals = [1.000, 0.875, 0.521, 0.833, 0.792, 0.875]
tom_vals  = [1.000, 0.818, 0.625, 0.917, 0.917, 0.781]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), subplot_kw=dict(polar=True))
fig.patch.set_facecolor("#0f0f0f")

pair_data = [
    ("Brad & Margot",   brad_vals,  "#ff6b6b", "#ff9999"),
    ("Tom & Jennifer",  tom_vals,   "#4ecdc4", "#7ee8e2"),
]

for ax, (title, scores, line_color, fill_color) in zip(axes, pair_data):
    vals = scores + [scores[0]]
    ax.set_facecolor("#1a1a2e")
    ax.set_theta_offset(math.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], fontsize=7, color="#555")
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(LABELS, fontsize=9, color="#ccc")
    ax.grid(color="#333", linewidth=0.5)
    ax.spines["polar"].set_color("#444")
    ax.plot(angles, vals, color=line_color, linewidth=2.5)
    ax.fill(angles, vals, color=fill_color, alpha=0.2)
    for angle, val in zip(angles[:-1], scores):
        ax.plot(angle, val, "o", color=line_color, markersize=6)
    total = compute_score(dict(zip(WEIGHTS.keys(), scores)))
    ax.set_title(f"{title}\n{total*100:.1f}% compatible",
                 fontsize=12, color="white", pad=20, fontweight="bold")

plt.suptitle("CBS Match — Compatibility Breakdown", fontsize=14, color="white",
             y=1.02, fontweight="bold")
plt.tight_layout()
plt.savefig("compatibility_radar.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ── Side-by-side bar comparison ───────────────────────────────────────────────

short_labels = ["Values", "Attachment", "Conflict", "Emotional", "Personality", "Life Goals"]
x     = np.arange(len(short_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("#0f0f0f")
ax.set_facecolor("#1a1a2e")

b1 = ax.bar(x - width/2, brad_vals, width, label="Brad & Margot",
            color="#ff6b6b", alpha=0.85, edgecolor="#cc2200")
b2 = ax.bar(x + width/2, tom_vals,  width, label="Tom & Jennifer",
            color="#4ecdc4", alpha=0.85, edgecolor="#1aaa99")

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f"{h*100:.0f}%", ha="center", va="bottom", fontsize=8, color="white")

ax.axhline(0.60, color="#888", linestyle="--", linewidth=1, label="Min threshold (60%)")
ax.set_xticks(x)
ax.set_xticklabels(short_labels, color="#ccc")
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"], color="#999")
ax.set_ylim(0, 1.15)
ax.set_title("Compatibility by Category", color="white", fontsize=13, fontweight="bold")
ax.legend(facecolor="#222", edgecolor="#444", labelcolor="white", fontsize=9)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("#444")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="#333", linewidth=0.5)

plt.tight_layout()
plt.savefig("compatibility_bars.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

**Reading the charts:**

- **Both pairs hit 100% values alignment** — they want fundamentally the same things from life and relationships.
- **Tom & Jennifer score 91–92% on both personality and emotional fit** — they're remarkably similar in how they experience and respond to the world.
- **Brad & Margot have a conflict-style gap** (52%) — Brad addresses tension immediately; Margot prefers to ease in gradually. The app surfaces this directly so they know going in.
- Every category for both pairs clears the 60% minimum — these are strong, stable matches.

---

## Part 9: Meet the Matches — Try It Live

The app is live on Vercel. Log in as any of the four users below to experience the full match reveal. **Log out and log in as the other half of the pair to see both perspectives.**

---

### Match 1 — Brad & Margot &nbsp;&nbsp; 80.3% compatible

**Brad** — Boston, CBS '26. Extroverted, career-driven, and direct. High conscientiousness and emotional stability. Wants to address conflict head-on. Ideal first date: cocktails.

**Margot** — San Francisco, CBS '26. Intellectually curious and spontaneous. The type to book a last-minute flight on a Thursday. Still figuring out the kids question, open to it. Ideal first date: museum or gallery.

Their strength is 100% values alignment and strong attachment compatibility. The conflict-style difference is real — the algorithm flags it, but the research says complementary communication styles often work fine once both people know the pattern.

```
Site:       https://cbs-match-web.vercel.app

Brad:       brad@gsb.columbia.edu      /  community123
Margot:     margot@gsb.columbia.edu    /  community123
```

---

### Match 2 — Tom & Jennifer &nbsp;&nbsp; 83.3% compatible

**Tom** — Chicago, CBS '26. Intellectually curious, open to new experiences, thoughtful and playful. Balanced on career vs. relationship. Values connection and conversation over social performance.

**Jennifer** — New York, CBS '26. Warm, socially grounded, relationship-forward. She checks in, she remembers things, she makes the group feel like family. High agreeableness and secure attachment.

This is the stronger match by score — nearly tied on every dimension. The algorithm ranks it as the most stable pairing from the full eligible pool.

```
Site:       https://cbs-match-web.vercel.app

Tom:        tom@gsb.columbia.edu         /  community123
Jennifer:   jennifer@gsb.columbia.edu    /  community123
```

---

### Experiencing the New User Flow

To see what it's like to sign up fresh — the survey, the vibe card, the profile setup — you can create a new account at **https://cbs-match-web.vercel.app/register** with any `@gsb.columbia.edu` address.

> If you don't have a `@gsb.columbia.edu` email, you can watch the flow by logging into one of the demo accounts above. The survey and vibe card are accessible from the profile settings page even after completion.

After completing the survey, you'll land on a "check back Monday" screen for your match — which is the real experience for new users. The matching algorithm runs once per week. The demo accounts skip this because we pre-ran the matching for this submission.

---

**Troubleshooting:** If the app isn't loading or login isn't working, the most likely cause is Render's free tier spinning down (~30s cold start). Refresh and try again. If the issue persists, reach out to Tom directly — tcline25@gsb.columbia.edu — and we'll fix it.

In [ ]:
# The algorithm also generates a human-readable explanation for each match.
# Here's what it says about Tom & Jennifer:

tom_token   = login("tom@gsb.columbia.edu")
data        = api_get("/matches/current", tom_token)
match       = data.get("match", {})
explanation = match.get("explanation") or {}

print("What the algorithm says about Tom & Jennifer:\n")
if isinstance(explanation, dict):
    for bullet in explanation.get("bullets", []):
        print(f"  • {bullet}")
    summaries = explanation.get("summary", [])
    if summaries:
        print(f"\n  Summary: {summaries[0]}")

---

## Part 10: What We Built, and What Actually Took the Time

### Working system as of this submission

| Feature | Status |
|---|---|
| Registration with CBS email validation | ✓ Live |
| JWT auth with silent token refresh | ✓ Live |
| 75-question personality survey with progress saving | ✓ Live |
| Vibe card generation on survey completion | ✓ Live |
| Weighted compatibility scoring (6 dimensions) | ✓ Live |
| Gale-Shapley stable matching, weekly cadence | ✓ Live |
| Match reveal with algorithm explanation | ✓ Live |
| Accept / decline with feedback | ✓ Live |
| Profile photos (upload from web app) | ✓ Live |
| Block and report safety features | ✓ Live |
| Admin dashboard (user management, match runs, analytics) | ✓ Live |
| Multi-tenant architecture (all M7 schools pre-configured) | ✓ Live |

### The bugs that cost us the most time

Honest accounting, because this is a class about learning Python:

1. **Survey version mismatch** — The backend config had `SURVEY_VERSION = 1` hardcoded, but the active survey in the production database was version 3. The matching query filtered for version 3, so every seeded user had version-1 sessions and zero appeared as eligible. Result: 20 users created, 0 matches. The fix was one line — but finding it took hours of API debugging.

2. **Cross-origin cookie failure** — Several pages called the Render backend directly from the browser. Auth cookies are `SameSite=Lax`, which means they're not sent on cross-origin `fetch()` calls. Every request returned 401. Fix: route all client-side API calls through Next.js proxy handlers on the same origin.

3. **Photo URLs using the internal address** — Without `--proxy-headers` on uvicorn, `request.base_url` returned `http://0.0.0.0:8000` instead of the public HTTPS URL. Photos uploaded fine but every URL stored in the database was unreachable. Fix: add `PUBLIC_API_URL` env var override.

### How AI accelerated this

Being honest about the workflow:

- **ChatGPT** was used as a research and review partner for the psychology layer — which Big Five dimensions predict long-term relationship satisfaction, how to weight the conflict vs. attachment dimensions, and whether the survey question selection was defensible against the literature. It was also used to review and refine the survey copy so questions felt natural rather than like a psych test.

- **Claude** (Anthropic) served as the primary coding collaborator for architecture decisions, Python backend implementation, debugging, and deployment configuration.

- **Claude Agents via [Cline](https://github.com/cline/cline)** (VS Code extension) did the actual building. Cline interfaces with Claude as an autonomous agent inside VS Code: it reads files, writes and edits code, runs terminal commands, and navigates the full codebase. We'd describe a feature or a bug, and the agent would implement it end-to-end — read the relevant files, write the change, run it, fix any errors, and commit. This is how three weeks of part-time work produced a full-stack production app. The human contribution was product thinking, design decisions, and judgment — the agent handled execution.

Total token usage across the build: **well over one billion tokens**. That number would have been inconceivable for a class project two years ago. It's what made this possible in the time we had.

---

## Part 11: What We'd Improve

**Immediate priorities**

- **Persistent photo storage** — Photos live on Render's ephemeral filesystem and are wiped on every redeploy. The fix is a Render Persistent Disk (mount at `/app/uploads`) or migrating uploads to Cloudflare R2. This is the highest-priority remaining issue.

- **Email notifications** — There's currently no way to tell users "your match dropped." Adding a transactional email layer (Resend or SendGrid) that fires Monday morning when matches are revealed would dramatically improve engagement.

- **Match expiry UX** — Matches expire after 72 hours. The current expired state is a bit abrupt. We'd add a countdown timer and a softer "this week's match has closed" screen.

**Longer term**

- **Feedback loop into the algorithm** — If two people both accept their match, collect a short follow-up a week later. Use that signal to tune the dimension weights over time. The current weights came from literature review; real CBS data would be better.

- **Richer profiles** — A 60-second voice note. A short written bio. The vibe questions on the survey already give us interesting material; surfacing more of that on the profile would help the match feel more personal before a first message.

- **Open enrollment across M7** — The infrastructure is in place. Activating Wharton, HBS, Booth, Kellogg, and Sloan is mostly a question of marketing and getting someone at each school to champion it. The code doesn't change.

---

## Conclusion

CBS Match started from a simple observation: [DateDrop worked for Stanford](https://stanford.trydatedrop.com/). Could we build something like that for CBS — but grounded in personality psychology rather than photos and swipes?

Three weeks and a billion tokens later, the answer is yes.

Python turned out to be an ideal foundation. FastAPI made the API layer expressive and fast to build. SQLAlchemy kept the database logic clean. The scoring engine and the Gale-Shapley implementation are both pure Python — they're readable, testable, and easy to tune. When we needed to fix a bug in production, the feedback loop between writing Python and seeing it live was measured in minutes.

The two matched pairs — **Brad & Margot** and **Tom & Jennifer** — are live right now on the production system. They can each see each other's profiles, their compatibility scores, and the algorithm's explanation for why they were paired. The system works.

We built this to submit for class. We also built it hoping someone at CBS actually uses it.

---

| Resource | Link |
|---|---|
| Live app | https://cbs-match-web.vercel.app |
| API + docs | https://cbs-match-api.onrender.com/docs |
| GitHub | https://github.com/T-J-C-99/cbs-match |
| Inspiration | https://stanford.trydatedrop.com/ |

*Built with: Python · FastAPI · SQLAlchemy · PyJWT · bcrypt · Next.js · PostgreSQL · Render · Vercel · Claude · ChatGPT · Cline*